In [26]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,stats_test,STT_INIT_POINTS, STT_N_ITER, VOT_INIT_POINTS, VOT_N_ITER
from scipy import stats

VOT config

In [27]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : defaultAlpha_bounds,
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-0.2,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.2),
           'weightWalk': (1,3),
           'weightWait': (1,3),
           'weightFeeder': (1,3), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [28]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : defaultAlpha_bounds,
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

In [29]:
stt_gen_synth_pop_baseline = [1.538359177843295,
 1.5295917791690679,
 1.5348367663998912,
 1.5345692372273643,
 1.5331592512372867,
 1.5217034952544148,
 1.5266326185511052,
 1.5334169124971047,
 1.5360137686306476,
 1.5339017221565026,
 1.5362097781749502,
 1.5268521109958817,
 1.5339477732745261,
 1.5272698871615034,
 1.5366142065675472,
 1.5363138577063769,
 1.5296998668830282,
 1.5197286106223606,
 1.5389714190118928,
 1.5313439434581202,
 1.537382448574118,
 1.5343533532165867,
 1.533794502058735,
 1.5379044402275153,
 1.540476837032657,
 1.5442550528088403,
 1.5314370912595778,
 1.5282256003186678,
 1.529788503142131,
 1.5278669293628444]

vot_gen_synth_pop_baseline = [1.385527667908908,
 1.3928263136401784,
 1.3789626118986595,
 1.3877472402566997,
 1.3871102011278413,
 1.385742628118824,
 1.3912868550950295,
 1.3872805327363473,
 1.3837930639491043,
 1.3895217968284068,
 1.3875468568014604,
 1.378848603328448,
 1.3894661348423816,
 1.388160109275813,
 1.3871204285727328,
 1.390915593919979,
 1.3787646467724206,
 1.384115813260929,
 1.3823050198239224,
 1.3847381859154229,
 1.3816234011419537,
 1.3867555230241397,
 1.3876604079221397,
 1.3822073266194879,
 1.3825751517648885,
 1.3835028373387503,
 1.3806274629010047,
 1.3837965494208504,
 1.389955262359422,
 1.377912990757942]

stt_base_pop_baseline = [8.078856646164647,
 8.069551887294185,
 8.075341126236319,
 8.077668179308883,
 8.072089261165532,
 8.08434725328873,
 8.070135626168561,
 8.07003999884651,
 8.074229639153051,
 8.073871153762488,
 8.076644709455191,
 8.067903638080667,
 8.074645258436167,
 8.07695697572408,
 8.069666279765226,
 8.073238056549226,
 8.075901298340282,
 8.074258667977556,
 8.066514722376704,
 8.072437841187186,
 8.07148870370448,
 8.069260779586159,
 8.067953580518305,
 8.074462078074248,
 8.072890298665042,
 8.076124680057342,
 8.07104170586535,
 8.066528599568002,
 8.070791356185456,
 8.081042173765757]

vot_base_pop_baseline = [7.931848576119684,
 7.931369714896462,
 7.9433848896215995,
 7.937736686895536,
 7.941242398531084,
 7.936923206985956,
 7.936669645073594,
 7.940973608526627,
 7.936207185980619,
 7.936291631277916,
 7.937923289418983,
 7.941331709310874,
 7.93954170300512,
 7.936830089802029,
 7.939676996588786,
 7.9394618950690505,
 7.939708733929694,
 7.941668216327173,
 7.941204779953899,
 7.943912118112101,
 7.944579406069159,
 7.941662315056593,
 7.933800474344775,
 7.935876751446223,
 7.9376907386207165,
 7.935850335717845,
 7.939215465650584,
 7.94108731535453,
 7.934474803549772,
 7.94179396788631]

In [30]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



RQ3.1

In [31]:

rq_3_1_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-1.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-1.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-1.csv',
    "other_args" : "--use_random_seed true"
}
rq3_1_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_3_1_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq3_1_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.02462 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -19.70015 | 0.0       | -3.018985 | 3.0074456 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -22.63785 | 0.0   

In [ ]:
rq_3_1_config_no_seed = rq_3_1_config
rq_3_1_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_1_results = []
for i in range(0,30):
    rq_3_1_results.append(-call_stt_simulation(config=rq_3_1_config_no_seed,**rq3_1_optimiser.max['params']))


In [33]:
rq_3_1_results

[7.84649316031477,
 7.839360148429676,
 7.838655220925358,
 7.842316619826978,
 7.837378055137598,
 7.8357478628903285,
 7.841468605876705,
 7.842859975200833,
 7.838427880781675,
 7.840354426065052,
 7.84088634902725,
 7.83834574997436,
 7.83887821240288,
 7.841471057051815,
 7.842648012183861,
 7.841180525265182,
 7.842577090715129,
 7.842228631893823,
 7.844653735286928,
 7.83916946602004,
 7.839550350605722,
 7.841056060629679,
 7.841041897991418,
 7.846077206375915,
 7.841326612420235,
 7.843402070043801,
 7.838479286515432,
 7.841479850458642,
 7.840362796574018,
 7.841098825020327]

RQ3.2

In [34]:

rq_3_2_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-2.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-2.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-2.csv',
    "other_args" : "--use_random_seed true"
}
rq_3_2_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_3_2_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq_3_2_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.90358 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.2      | -0.2      | -0.2      | 1.8383890 | 2.3704390 | 1.4089044 | 1.0       | 1.0       |
| 2         | -15.08848 | 0.0       | -0.826951 | 0.5868982 | -3.596130 | -3.018985 | 3.0074456 | -0.032706 | -0.2      | -0.2      | -0.2      | 2.7892133 | 1.1700884 | 1.0781095 | 1.0       | 1.0       |
| 3         | -10.47610 | 0.0       | -0.788923 | 4.5788953 | 0.3316528 | 1.9187711 | -1.844843 | -0.314185 | -0.2      | -0.2      | -0.2      | 2.9777221 | 2.4963313 | 1.5608

In [35]:
rq_3_2_config_no_seed = rq_3_2_config
rq_3_2_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_2_results = []
for i in range(0,30):
    rq_3_2_results.append(- call_vot_simulation(config=rq_3_2_config_no_seed,**rq_3_2_optimiser.max['params']))

In [36]:
rq_3_2_results

[7.7505706563748955,
 7.754477018931189,
 7.7495401327862155,
 7.757613238718088,
 7.7534913930546026,
 7.7537377854406175,
 7.756086180704962,
 7.74619042557366,
 7.756006077592342,
 7.751169604737998,
 7.754513410328947,
 7.757741212161856,
 7.754354196784257,
 7.754233228052612,
 7.754181987381714,
 7.754197237572841,
 7.757238140196032,
 7.752809932296534,
 7.758236883627914,
 7.744535576378034,
 7.751056656733643,
 7.752406099644705,
 7.751239332447279,
 7.753988464501633,
 7.7534096102151855,
 7.750341563996585,
 7.752838877941095,
 7.751089408785668,
 7.753396192469911,
 7.7536990308677165]

RQ3.3

In [37]:

rq_3_3_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-3.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-3.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-3.csv',
    "other_args" : "--use_random_seed true"
}
rq3_3_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_3_3_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq3_3_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.11254 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -18.15768 | 0.0       | -3.018985 | 3.0074456 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -21.94948 | 0.0   

In [38]:
rq_3_config_no_seed = rq_3_3_config
rq_3_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_3_results = []
for i in range(0,30):
    rq_3_3_results.append(-call_stt_simulation(config=rq_3_config_no_seed,**rq3_3_optimiser.max['params']))

In [39]:
rq_3_3_results

[1.4815916890729006,
 1.473860153787428,
 1.4772560434824866,
 1.4727220907126557,
 1.4724838821276645,
 1.4688630280469501,
 1.4731057855381455,
 1.4763374597361807,
 1.479730751741111,
 1.4828397909061828,
 1.4804068379067457,
 1.4699810185733058,
 1.4721610884796352,
 1.4800888522905689,
 1.4647087961643777,
 1.476351237468731,
 1.4751664089760628,
 1.4716929501435867,
 1.4740823129690406,
 1.4788289768180296,
 1.4747243874212381,
 1.479206567639927,
 1.4736095204483637,
 1.478317589070644,
 1.4763082213092242,
 1.4837715788057226,
 1.4788477726736697,
 1.4764108611903548,
 1.4712103279953317,
 1.474831813244142]

RQ3.4

In [40]:
rq_3_4_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-4.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-4.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-4.csv',
    "other_args" : "--use_random_seed true"
}
rq_3_4_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_3_4_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq_3_4_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.70852 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.2      | -0.2      | -0.2      | 1.8383890 | 2.3704390 | 1.4089044 | 1.0       | 1.0       |
| 2         | -12.16617 | 0.0       | -0.826951 | 0.5868982 | -3.596130 | -3.018985 | 3.0074456 | -0.032706 | -0.2      | -0.2      | -0.2      | 2.7892133 | 1.1700884 | 1.0781095 | 1.0       | 1.0       |
| 3         | -5.582873 | 0.0       | -0.788923 | 4.5788953 | 0.3316528 | 1.9187711 | -1.844843 | -0.314185 | -0.2      | -0.2      | -0.2      | 2.9777221 | 2.4963313 | 1.5608

In [41]:
rq_3_4_config_no_seed = rq_3_4_config
rq_3_4_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_4_results = []
for i in range(0,30):
    rq_3_4_results.append(- call_vot_simulation(config=rq_3_4_config_no_seed,**rq_3_4_optimiser.max['params']))

In [42]:
rq_3_4_results

[1.3753081319569687,
 1.3742488303399583,
 1.368884010723444,
 1.3901922800580793,
 1.361084756061178,
 1.3775440627186848,
 1.3790867291461066,
 1.3727054733959452,
 1.365534949443592,
 1.3863956306982672,
 1.369942893561297,
 1.3627206632717621,
 1.3818975650700178,
 1.3661656227329662,
 1.3582229687456173,
 1.3829006610569607,
 1.371984528786159,
 1.379002606109665,
 1.3698603823393452,
 1.3873749242135547,
 1.376056809933175,
 1.388874581056718,
 1.3668109468143579,
 1.3580711757372352,
 1.3754509132818438,
 1.3793868968859846,
 1.3736371235050056,
 1.3867188912473,
 1.3659524784652293,
 1.3679807519092755]

Significance

In [43]:
stats_test(stt_base_pop_baseline, rq_3_1_results)

P-value: 6.257103952723728e-91
Statistically significant difference.
t-statistic: 262.9329
p-value: 0.0000
95% CI of the difference: [0.2305, 0.2340]


In [44]:
stats_test(vot_base_pop_baseline, rq_3_2_results)

P-value: 1.0720496155561564e-86
Statistically significant difference.
t-statistic: 222.2164
p-value: 0.0000
95% CI of the difference: [0.1838, 0.1872]


In [45]:
stats_test(stt_gen_synth_pop_baseline, rq_3_3_results)

P-value: 5.504833186531317e-47
Statistically significant difference.
t-statistic: 45.3335
p-value: 0.0000
95% CI of the difference: [0.0546, 0.0597]


In [46]:
stats_test(vot_gen_synth_pop_baseline, rq_3_4_results)

P-value: 5.312776468208588e-08
Statistically significant difference.
t-statistic: 6.2472
p-value: 0.0000
95% CI of the difference: [0.0077, 0.0149]
